## MLS WS 2025/26 Exercise 3d: Spam Email Classification with Machine Learning and Deep Learning (39 Points)
*Adapted from an exercise created by Dennis Eisermann*

This notebook introduces basic concepts of email filtering. Your task is to build a robust spam email classifier using a [combined dataset of emails](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset). This dataset uses the labels 1 for spam and 0 for ham. The dataset contains the columns 'text_combined', which is the email content and 'label'.

#### Learning Goals:
<ul>
<li>Understand text data preprocessing techniques (TF-IDF, tokenization).</li>
<li>Perform exploratory data analysis (EDA) on text data.</li>
<li>Train and evaluate traditional ML models (e.g., Logistic Regression, Random Forest).</li>
<li>Compare model performance using metrics like accuracy, precision, recall, and F1-score.</li>
<li>Interpret confusion matrices to identify model weaknesses.</li>
</ul>

### Task 1 – Setup and Preprocessing (3 Points)



1. Install and import the necessary libraries: pandas, numpy, matplotlib, seaborn and sklearn.

In [1]:
!pip install pandas matplotlib seaborn numpy nltk xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.1 MB/s  0:00:00
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
    --------------------------------------- 1.0/72.0 MB 6.5 MB/s eta 0:00:11
   - -------------------------------------- 2.6/72.0 MB 6.6 MB/s eta 0:00:11
   - -------------------------------------- 3.4/72.0 MB 6.1 MB/s eta 0:00:12
   -- ------------------------------------- 4.7/72.0 MB 6.0 MB/s eta 0:00:12
   --- ------------------------------------ 6.0/72.0 MB 6.0 MB/s eta 0:00:12
   --- ------------------------------------ 7.1/72.0 MB 6.0 MB/s eta 0:00:11
   ---- ----------------------------------- 8.1/72.0 MB 5.9 MB/s eta 0:00:11
   ----- ---------------------------------- 9.2/72.0 MB 5.8 MB/s eta 0:00:11
   ----- ---------

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer


2. Load the dataset into a pandas Dataframe. (1 Point)

In [11]:
import pandas as pd
import zipfile

# ZIP-Datei öffnen und alles extrahieren
with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall("data")  # entpackt alle Dateien in den Ordner "data"

# Liste der enthaltenen CSV-Dateien
csv_files = [
    "CEAS_08.csv",
    "Enron.csv",
    "Ling.csv",
    "Nazario.csv",
    "Nigerian_Fraud.csv",
    "SpamAssasin.csv",
    "phishing_email.csv"
]

# Dictionary für alle DataFrames
dataframes = {}

# Jede CSV laden und im Dictionary speichern
for file in csv_files:
    path = f"data/{file}"
    df = pd.read_csv(path)
    dataframes[file] = df
    print(f"--- {file} ---")
    print(df.head(), "\n")

# Zugriff auf ein bestimmtes DataFrame:
# Beispiel: Enron-Datensatz
CEAS_08_df = dataframes["CEAS_08.csv"]
enron_df = dataframes["Enron.csv"]
Ling_df = dataframes["Ling.csv"]
Nazario_df = dataframes["Nazario.csv"]
Nigerian_Fraud_df = dataframes["Nigerian_Fraud.csv"]
phishing_email_df = dataframes["phishing_email.csv"]


--- CEAS_08.csv ---
                                              sender  \
0                   Young Esposito <Young@iworld.de>   
1                       Mok <ipline's1983@icable.ph>   
2  Daily Top 10 <Karmandeep-opengevl@universalnet...   
3                 Michael Parker <ivqrnai@pobox.com>   
4  Gretchen Suggs <externalsep1@loanofficertool.com>   

                                         receiver  \
0                     user4@gvc.ceas-challenge.cc   
1                   user2.2@gvc.ceas-challenge.cc   
2                   user2.9@gvc.ceas-challenge.cc   
3  SpamAssassin Dev <xrh@spamassassin.apache.org>   
4                   user2.2@gvc.ceas-challenge.cc   

                              date  \
0  Tue, 05 Aug 2008 16:31:02 -0700   
1  Tue, 05 Aug 2008 18:31:03 -0500   
2  Tue, 05 Aug 2008 20:28:00 -1200   
3  Tue, 05 Aug 2008 17:31:20 -0600   
4  Tue, 05 Aug 2008 19:31:21 -0400   

                                             subject  \
0                          Never agree 

3. Encode the label column by replacing 0 with 'ham' and 1  with 'spam'. Print the header of the result. (1 Point)

In [ ]:
# Beispiel: wir nehmen an, die Spalte heißt 'label'
CEAS_08_df['label'] = CEAS_08_df['label'].replace({0: 'ham', 1: 'spam'})
enron_df['label'] = enron_df['label'].replace({0: 'ham', 1: 'spam'})
Ling_df['label'] = Ling_df['label'].replace({0: 'ham', 1: 'spam'})
Nazario_df['label'] = Nazario_df['label'].replace({0: 'ham', 1: 'spam'})
Nigerian_Fraud_df['label'] = Nigerian_Fraud_df['label'].replace({0: 'ham', 1: 'spam'})
phishing_email_df['label'] = phishing_email_df['label'].replace({0: 'ham', 1: 'spam'})

# Alternativ:
# df['label'] = df['label'].map({0: 'ham', 1: 'spam'})

# Header (erste Zeilen) ausgeben
print(df.head())


                                       text_combined label
0  hpl nom may 25 2001 see attached file hplno 52...   ham
1  nom actual vols 24 th forwarded sabrae zajac h...   ham
2  enron actuals march 30 april 1 201 estimated a...   ham
3  hpl nom may 30 2001 see attached file hplno 53...   ham
4  hpl nom june 1 2001 see attached file hplno 60...   ham


4. Prepare training and test data for later use. Split the data into 70% training and 30% testing sets. Set 42 as random seed for reproducibility. Check for missing values and handle them if necessary. (1 Point)

In [ ]:
# Beispiel: wir nehmen an, die Daten sind schon in df geladen
# Prüfen auf fehlende Werte
print("Missing values per column:")
print(df.isnull().sum())

# Falls fehlende Werte vorhanden sind → einfache Behandlung:
# Möglichkeit 1: Entfernen
df = df.dropna()

# Möglichkeit 2: Auffüllen (z.B. mit leeren Strings oder 0)
# df = df.fillna("")

# Features und Label trennen
X = df.drop("label", axis=1)   # alle Spalten außer 'label'
y = df["label"]                # Zielspalte

# Split in Training (70%) und Test (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


### Task 2 – Data Analysis (12 Points)

Analyse the data bit in order to find out how the dataset is structured, and to see how text data can be processed to filter for the meaningful features later.

1. Calculate and visualize the class distribution (spam vs. ham) using a bar plot. (1 Point)

2. Compute the average word count per email for spam and ham. Plot the distributions as bar plots. (2 Points)

3. Identify the top 10 most frequent words in spam emails by following the process below. (overall 4 Points)

   It is advisable to join the spam emails into one string.

   So-called stopwords carry little semantic meaning and removing them improves performance and results.
   Use the stopword list from the [nltk package](https://www.geeksforgeeks.org/python/introduction-to-nltk-tokenization-stemming-lemmatization-pos-tagging/) to remove stop words (2 Points).

   Calculate the frequencies for the remaining words and print out the 10 most frequent ones together with their number of occurrence. (2 Points)

In [ ]:
import nltk
nltk.download('stopwords')



4. Repeat the previous steps for the 'ham' emails. (1 Point)

5. Describe in two sentences which differences you see when comparing your results for the most frequent words in spam and ham emails. (1 Point)

*your answer*

6. Visualize the distribution of email lengths (character count) for both classes. Use a KDE Plot from the [seaborn package](https://www.geeksforgeeks.org/r-data-visualization/kde-plot-visualization-with-pandas-and-seaborn/) for the plots. (3 Points)

### Task 3 – Feature Preparation (6 Points)

In this task, you prepare the data for later use in training and testing. Therefore, filter out unnecessary stopwords and punctuation and transform the data.

1. The first step is to preprocess the text of the emails. Write a function to convert it to lowercase, remove punctuation, tokenize it (see useful nltk functions for this) and remove the stopwords for each mail in the dataset. Use the stopword list from the [nltk package](https://www.geeksforgeeks.org/python/introduction-to-nltk-tokenization-stemming-lemmatization-pos-tagging/). (3 Points)

2. Write a function which uses [TfidfVectorizer](https://www.geeksforgeeks.org/understanding-tf-idf-term-frequency-inverse-document-frequency/) to create TF-IDF features (max_features=5000). The matrix of the vectorizer can directly be used as output and does not have to be converted to a Dataframe or array! (2 Points)

3. Use the function to transform the training and testing data. (1 Point)

### Task 4 – Model Training (6 Points)

Train three different classifiers to distinguish between ham and spam.

1. Train a Logistic Regression model using the TF-IDF features. (2 Points)

2. Train a Random Forest model with 100 estimators. (2 Points)

3. Train an [XGBoost classifier](https://xgboost.readthedocs.io/en/stable/get_started.html). (2 Points)


### Task 5 – Evaluation (12 Points)

In this task you evaluate the performances of the three models with several different statistics and draw conclusions.

1. Predict the labels of the test set for all three models. (1 Point)

2. Compute accuracy, precision, recall, F1-score, and ROC-AUC for all three models using 'sklearn.metrics'. Print your results. (3 Point)

3. Generate confusion matrices for all models. Plot them as heatmaps. (3 Points)

4. Compare ROC-AUC scores across all models using a bar plot. (1 Point)

5. Identify which model has the highest precision and explain shortly why this matters for spam detection. (2 Points)

*your answer*

6. Interpret the confusion matrix for the best model: (2 Points)

    * Name the types of errors that are most common (false positives vs. false negatives).
    * Suggest improvements to reduce these errors.

*your answer*